# 🌿 Tutorial 1: Quickstart & First-Principles Thermodynamics

Welcome to the **BIOPLANT AI** Digital Twin interactive notebook series!
In this tutorial, you will:
1. Initialize the digital twin simulation environment.
2. Load calibrated biomass feedstocks (**Olive Pomace, Pine Sawdust, Wheat Straw, Rice Husk**).
3. Run first-principles thermodynamic simulations across reactor temperatures ($400^\circ\text{C} - 650^\circ\text{C}$).
4. Inspect mass closures, elemental balances ($C, H, O, N, S$), and the **Thermal Self-Sufficiency Index (TSI)**.

In [ ]:
import sys
from pathlib import Path

# Ensure repo root is in Python path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.simulation.plant_simulator import BiomassPlantSimulator
from src.data.preprocessing import FeedstockLibrary

print("[*] BIOPLANT AI simulation engine loaded successfully.")

## 1. Inspecting Feedstock Chemical Fingerprints

In [ ]:
lib = FeedstockLibrary()
for name in ["pine_sawdust", "olive_pomace", "wheat_straw", "rice_husk"]:
    fs = lib.load_feedstock(name)
    print(f"=== {fs.name} ({fs.category.upper()}) ===")
    print(f"  Proximate (dry wt%): Volatiles={fs.proximate.volatile_matter}%, Fixed Carbon={fs.proximate.fixed_carbon}%, Ash={fs.proximate.ash}%")
    print(f"  Ultimate (dry wt%) : C={fs.ultimate.carbon}%, H={fs.ultimate.hydrogen}%, O={fs.ultimate.oxygen}%, N={fs.ultimate.nitrogen}%, S={fs.ultimate.sulfur}%")
    print(f"  Heating Values     : HHV={fs.calculate_hhv_dry():.2f} MJ/kg | LHV={fs.calculate_lhv_dry():.2f} MJ/kg\n")

## 2. Running a Baseline Flowsheet Simulation
Let's simulate fast pyrolysis of **Olive Pomace** at $500^\circ\text{C}$ with an infeed of $100\text{ kg/h}$.

In [ ]:
simulator = BiomassPlantSimulator()
report = simulator.run_simulation(
    feedstock_name="olive_pomace",
    feed_rate_kg_h=100.0,
    reactor_temp_c=500.0,
    moisture_pct=12.0,
    heating_rate_c_min=10.0,
    residence_time_min=20.0,
    yield_mode="DETERMINISTIC"
)

print(f"=== Simulation Results: {report.feedstock.name} ===")
print(f"Bio-Oil Yield (dry wt%): {report.reactor.yields_dry.bio_oil_yield * 100:.2f}%")
print(f"Biochar Yield (dry wt%) : {report.reactor.yields_dry.biochar_yield * 100:.2f}%")
print(f"Syngas Yield (dry wt%)  : {report.reactor.yields_dry.syngas_yield * 100:.2f}%")
print(f"Recovered Bio-Oil Rate  : {report.separation.recovered_bio_oil_liquid_kg_h:.2f} kg/h")
print(f"Thermal Self-Sufficiency: {report.combustion.thermal_self_sufficiency_index_pct:.1f}% (Autonomous: {report.combustion.is_thermally_self_sufficient})")
print(f"Mass Balance Closure    : {report.mass_balance.closure_pct:.2f}%")
print(f"Carbon Balance Closure  : {report.elemental_balance.closures['C'].closure_pct:.2f}%")

## 3. Parametric Temperature Sweep & Self-Sufficiency Curve

In [ ]:
temps = [400, 450, 500, 550, 600, 650]
oil_yields = []
char_yields = []
tsi_values = []

for t in temps:
    rep = simulator.run_simulation(
        feedstock_name="olive_pomace",
        feed_rate_kg_h=100.0,
        reactor_temp_c=t,
        moisture_pct=12.0,
    )
    oil_yields.append(rep.reactor.yields_dry.bio_oil_yield * 100)
    char_yields.append(rep.reactor.yields_dry.biochar_yield * 100)
    tsi_values.append(rep.combustion.thermal_self_sufficiency_index_pct)

print("Temperature Sweep Summary:")
for t, oil, char, tsi in zip(temps, oil_yields, char_yields, tsi_values):
    print(f"  T={t}°C -> Bio-Oil: {oil:.1f}% | Biochar: {char:.1f}% | TSI: {tsi:.1f}%")